[Guide Index](README.md)

# 02d The Oxigraph-backed graph (`StarLayerGraph` / `StarLayerDataset`)

`StarLayerGraph` and `StarLayerDataset` wrap a `pyoxigraph.Store`. See `docs/working/oxgraph-wrapper.md` for the design. This guide covers the boundary between the wrapper and Oxigraph: what the wrapper does for you, what you reach through `.store`, and where behaviour differs from calling Oxigraph directly.

pyoxigraph's own classes cannot be subclassed, so the wrapper holds the store rather than extending it. Anything not listed here is documented by [pyoxigraph](https://pyoxigraph.readthedocs.io/).

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse variables from earlier ones. The guide covers what `StarLayerGraph` and `StarLayerDataset` are, then the helpers that go with them (terms, lists, changes, `profile()`, prefixes, imports); the data they work on is built in each section.

## 1. Basics

In [1]:
from pyoxigraph import NamedNode, Literal
from starlayer.graph import StarLayerGraph, StarLayerDataset, as_store

g = StarLayerGraph(prefixes={"ex": "http://example.org/"})
g.parse(data='''@prefix ex: <http://example.org/> .
ex:a ex:knows ex:b ; ex:age 4 .
ex:b ex:knows ex:c .''', format="turtle")
print(len(g), "triples")
print(g.serialize(format="turtle"))

3 triples
@prefix ex: <http://example.org/> .
ex:b ex:knows ex:c .
ex:a ex:age 4 ;
	ex:knows ex:b .



## 2. Querying

`query()` sends the SPARQL text straight to Oxigraph. `initNs` becomes Oxigraph's per-call `prefixes=`, and `initBindings` becomes `substitutions=`, which Oxigraph only applies to a variable that the query projects. The result is a `Result` wrapper that iterates rows.

In [2]:
EX = "http://example.org/"
result = g.query("SELECT ?s ?o WHERE { ?s ex:knows ?o }", initNs={"ex": EX})
for row in result:
    print(row["s"], "knows", row["o"])

bound = g.query("SELECT ?s ?o WHERE { ?s ex:knows ?o }", initNs={"ex": EX}, initBindings={"s": NamedNode(EX + "a")})
print([str(r["o"]) for r in bound])

<http://example.org/b> knows <http://example.org/c>
<http://example.org/a> knows <http://example.org/b>
['<http://example.org/b>']


## 3. The store boundary: `.store` and `as_store()`

`g.store` is the underlying `pyoxigraph.Store`; `as_store(x)` returns it for a wrapper and passes a `Store` through unchanged. Seven `Store` methods are also wrapped (`add`, `clear`, `extend`, `query`, `remove`, `remove_graph`, `update`). The rest are reached through `.store`. The wrapper does **not** forward unknown attributes, so `g.flush()` is an `AttributeError`.

In [3]:
import pyoxigraph as ox

store = g.store
print(type(store).__name__, as_store(g) is store)
wrapped = {"add", "clear", "extend", "query", "remove", "remove_graph", "update"}
public = {m for m in dir(ox.Store) if not m.startswith("_")}
print("only through .store:", sorted(public - wrapped))

Store True
only through .store: ['add_graph', 'backup', 'bulk_extend', 'bulk_load', 'clear_graph', 'contains_named_graph', 'dump', 'flush', 'load', 'named_graphs', 'optimize', 'quads_for_pattern', 'read_only']


In [4]:
try:
    g.flush()
except AttributeError as error:
    print("AttributeError:", error)
store.flush()  # persistence operations go through the store

AttributeError: 'StarLayerGraph' object has no attribute 'flush'


## 4. Persistence

A path opens an on-disk (RocksDB) store. Only one process may write to it at a time, and there are no transactions beyond the atomicity of a single `update`.

In [5]:
import tempfile

path = tempfile.mkdtemp()
disk = StarLayerGraph(path=path)
disk.add((NamedNode(EX + "s"), NamedNode(EX + "p"), Literal("kept")))
disk.store.flush()
print(len(disk), "triple on disk")

1 triple on disk


## 5. Set operations

The operators `+`, `-`, `&` and `^` work on wrappers; `starlayer.graph.setops` offers the same operations as functions that also accept bare `pyoxigraph` stores. Blank nodes are compared by label, not by isomorphism.

In [6]:
from starlayer.graph import setops

other = StarLayerGraph()
other.add((NamedNode(EX + "a"), NamedNode(EX + "knows"), NamedNode(EX + "b")))
print("difference:", len(g - other), " intersection:", len(g & other), " union:", len(g + other), " symmetric:", len(g ^ other))
print("on bare stores:", len(setops.difference(g.store, other.store)))

difference: 2  intersection: 1  union: 3  symmetric: 2
on bare stores: 2


## 6. Where Oxigraph differs from rdflib

**The store rewrites some literals on insert** (issue #190). A lexical form such as `"04"^^xsd:integer` is stored as `4`; the original text is not recoverable from the store. Ill-formed values and strings are kept as written.

In [7]:
h = StarLayerGraph()
h.parse(data='<http://e/a> <http://e/age> "04"^^<http://www.w3.org/2001/XMLSchema#integer> .', format="turtle")
print([str(o) for o in h.objects()])

['"4"^^<http://www.w3.org/2001/XMLSchema#integer>']


**RDF 1.1 output.** The wrapper stores RDF 1.2 terms, including triple terms. To hand a graph to an RDF 1.1 consumer, serialise with `rdf_version="1.1"`; triple terms become the "basic encoding" (`rdf:PropositionForm` and its subject, predicate and object properties).

In [8]:
t = StarLayerGraph()
t.parse(data='''PREFIX ex: <http://example.org/>
ex:a ex:says <<( ex:b ex:knows ex:c )>> .''', format="turtle")
print(t.serialize(format="turtle", rdf_version="1.1"))

@prefix ex: <http://example.org/> .
ex:a ex:says _:ff90998f4be0071684a95682e8d0b395 .
_:ff90998f4be0071684a95682e8d0b395 a <http://www.w3.org/1999/02/22-rdf-syntax-ns#PropositionForm> ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormSubject> ex:b ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormPredicate> ex:knows ;
	<http://www.w3.org/1999/02/22-rdf-syntax-ns#propositionFormObject> ex:c .



`starlayer.graph.interop` does the same conversion in memory, on quads. `to_basic(quads)` replaces each triple term with a blank node typed `rdf:PropositionForm` that has `rdf:subject`, `rdf:predicate` and `rdf:object` properties; `from_basic(quads)` finds that structure and rebuilds the triple term; `has_triple_terms(quads)` says whether there is one to replace. `to_rdf11(graph)` also encodes base-direction literals (`direction_policy="datatype"`, `"error"` or `"drop"`), and `export_rdf11(...)` writes the result in an RDF 1.1 format.

In [9]:
from starlayer.graph import interop

quads = list(t.store)
print("has a triple term:", interop.has_triple_terms(quads))
basic = interop.to_basic(quads)
print(len(quads), "quad ->", len(basic), "quads in the basic encoding; a triple term left:", interop.has_triple_terms(basic))
restored = interop.from_basic(basic)
print("round trip restores it:", {(q.subject, q.predicate, q.object) for q in restored} == {(q.subject, q.predicate, q.object) for q in quads})

has a triple term: True
1 quad -> 5 quads in the basic encoding; a triple term left: False
round trip restores it: True


## 7. Datasets

`StarLayerDataset` holds named graphs in one store.

In [9]:
d = StarLayerDataset()
d.parse(data='<http://e/s> <http://e/p> <http://e/o> <http://e/g> .', format="nquads")
print(len(d), "quad;", "store type:", type(d.store).__name__)

1 quad; store type: Store


## 8. Terms and Python values

Terms are pyoxigraph terms (`NamedNode`, `BlankNode`, `Literal`, `Triple`). Three functions in `starlayer.graph` convert between them and Python. `literal(value, datatype=None, lang=None, direction=None)` makes a literal: a `bool` is `xsd:boolean`, an `int` `xsd:integer`, a `float` `xsd:double`, a `Decimal` `xsd:decimal`, and a date or time its `xsd` type. `to_python(term)` reads one back, and raises `ValueError` for a lexical form that is not valid for its datatype. `parse_term(text, prefixes=None)` reads exactly one term of Turtle 1.2 text (an IRI, a blank node, a literal, a triple term); a prefixed name needs `prefixes`.

In [10]:
import datetime
from decimal import Decimal

from starlayer.graph import XSD, literal, parse_term, to_python

for value in (True, 5, 2.5, Decimal("1.50"), datetime.date(2026, 1, 2), "text"):
    term = literal(value)
    print(repr(value), "->", term, "->", repr(to_python(term)))

print(literal("5", XSD.long), literal("hej", lang="sv"), literal("مرحبا", lang="ar", direction="rtl"))
print(parse_term('"4"^^xsd:integer', {"xsd": str(XSD)}), parse_term("<http://example.org/a>"), parse_term("<<( <urn:s> <urn:p> <urn:o> )>>"))

True -> "true"^^<http://www.w3.org/2001/XMLSchema#boolean> -> True
5 -> "5"^^<http://www.w3.org/2001/XMLSchema#integer> -> 5
2.5 -> "2.5"^^<http://www.w3.org/2001/XMLSchema#double> -> 2.5
Decimal('1.50') -> "1.50"^^<http://www.w3.org/2001/XMLSchema#decimal> -> Decimal('1.50')
datetime.date(2026, 1, 2) -> "2026-01-02"^^<http://www.w3.org/2001/XMLSchema#date> -> datetime.date(2026, 1, 2)
'text' -> "text" -> 'text'
"5"^^<http://www.w3.org/2001/XMLSchema#long> "hej"@sv "مرحبا"@ar--rtl
"4"^^<http://www.w3.org/2001/XMLSchema#integer> <http://example.org/a> <urn:s> <urn:p> <urn:o>


## 9. Lists, single values and walks

`graph.items(head)` reads an RDF list. `graph.add_list(items)` writes one on fresh blank nodes and returns its head (`rdf:nil` for no items); `graph.set_list(subject, predicate, items)` makes the items the one list a node has for a predicate, taking the cells of the old list away; `graph.remove_list(head)` removes the cells of a list (its members and the triple that points at it stay). `graph.set(triple)` makes a triple the only one with its subject and predicate. `graph.transitive_objects(subject, predicate)` and `graph.transitive_subjects(predicate, object)` follow a predicate until nothing new is reached, each node once and the start node first, so a cycle ends.

In [11]:
from starlayer.graph import RDFS, Namespace, StarLayerGraph

EXL = Namespace("http://example.org/")
lg = StarLayerGraph()
lg.set_list(EXL.order, EXL.steps, [EXL.fetch, EXL.parse, literal(3)])
print([str(t) for t in lg.items(lg.value(EXL.order, EXL.steps))])
lg.set_list(EXL.order, EXL.steps, [EXL.fetch])  # replaces the list: the old cells are gone
print(len(lg), "triples:", [str(t) for t in lg.items(lg.value(EXL.order, EXL.steps))])

lg.set((EXL.order, EXL.label, literal("first")))
lg.set((EXL.order, EXL.label, literal("second")))  # the only label now
print([str(o) for o in lg.objects(EXL.order, EXL.label)])

for sub, sup in [(EXL.Dog, EXL.Mammal), (EXL.Mammal, EXL.Animal), (EXL.Animal, EXL.Dog)]:  # a cycle
    lg.add((sub, RDFS.subClassOf, sup))
print([str(n) for n in lg.transitive_objects(EXL.Dog, RDFS.subClassOf)])
print([str(n) for n in lg.transitive_subjects(RDFS.subClassOf, EXL.Animal)])

['<http://example.org/fetch>', '<http://example.org/parse>', '"3"^^<http://www.w3.org/2001/XMLSchema#integer>']
3 triples: ['<http://example.org/fetch>']
['"second"']
['<http://example.org/Dog>', '<http://example.org/Mammal>', '<http://example.org/Animal>']
['<http://example.org/Animal>', '<http://example.org/Mammal>', '<http://example.org/Dog>']


## 10. Changes between graphs

`diff(old, new)` is a `GraphDelta`: the triples to add and the triples to remove to turn `old` into `new`. `delta.apply(graph)` makes the change and `delta.inverse()` undoes it; `a.then(b)` is the net effect of `a` followed by `b`; `delta.applied_to(graph)` says what the triples would be without changing the graph. Triples are compared as written, so a blank node is the same node when it has the same label: to ask whether two graphs are the same up to blank-node names use `isomorphic` ([6.c](06c-canonical-hashing.ipynb)).

In [12]:
from starlayer.graph import diff

old, new = StarLayerGraph(prefixes={"ex": EXL}), StarLayerGraph(prefixes={"ex": EXL})
old.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p ex:b ; ex:q ex:b .", format="turtle")
new.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p ex:c ; ex:q ex:b .", format="turtle")

delta = diff(old, new)
show = lambda triples: sorted(" ".join(str(t) for t in triple) for triple in triples)
print("add:", show(delta.added))
print("remove:", show(delta.removed))
delta.apply(old)
print("old now equals new:", set(old.triples()) == set(new.triples()))
delta.inverse().apply(old)
print("and back:", sorted(str(o) for o in old.objects(EXL.a, EXL.p)))

add: ['<http://example.org/a> <http://example.org/p> <http://example.org/c>']
remove: ['<http://example.org/a> <http://example.org/p> <http://example.org/b>']
old now equals new: True
and back: ['<http://example.org/b>']


## 11. What a graph contains

`graph.profile()` (or `profile(graph)`) reads the graph once and says what is in it: `classes` (each class with its number of instances), `properties` (each predicate with its number of triples) and, for the instances of a class or any set of nodes, the properties they use as `PropertyUse(property, nodes, max_values, datatypes, kinds)`: how many of the nodes have the property, the most values one node has, the datatypes of its literal values and the kinds of value (`"iri"`, `"blank"`, `"literal"`, `"triple"`). It is a snapshot: take another when the graph changes. A dataset is profiled by its `triples()`, which is the union of its graphs when `default_union` is on.

In [13]:
pg = StarLayerGraph()
pg.parse(data='''@prefix ex: <http://example.org/> .
ex:a a ex:Person ; ex:name "Ann" ; ex:age 30 ; ex:knows ex:b .
ex:b a ex:Person ; ex:name "Bob", "Robert" .''', format="turtle")
profile = pg.profile()
short = lambda term: term.value.rsplit("/", 1)[-1].rsplit("#", 1)[-1]
print({short(cls): count for cls, count in profile.classes.items()})
for use in profile.class_properties(EXL.Person):
    print(short(use.property), "nodes:", use.nodes, "most values:", use.max_values, "kinds:", sorted(use.kinds), "datatypes:", sorted(short(d) for d in use.datatypes))

{'Person': 2}
name nodes: 2 most values: 2 kinds: ['literal'] datatypes: ['string']
type nodes: 2 most values: 1 kinds: ['iri'] datatypes: []
age nodes: 1 most values: 1 kinds: ['literal'] datatypes: ['integer']
knows nodes: 1 most values: 1 kinds: ['iri'] datatypes: []


## 12. Prefixes

A new graph starts with no prefixes (`StarLayerGraph(prefixes={...})` gives it some). `graph.bind(prefix, namespace)` adds one (`override=True` replaces a prefix that is bound), `graph.unbind(prefix)` removes it, `graph.namespaces()` lists them, and `graph.used_prefixes()` lists the ones the graph's IRIs actually use. `graph.expand_curie("ex:alice")` makes the IRI of a prefixed name, `graph.compute_qname(iri)` splits an IRI into `(prefix, namespace, local name)` (a `KeyError` with `generate=False` when no prefix is bound), and `graph.qname_term(term)` writes a term as Turtle would with the graph's prefixes. To change what every new graph starts with, `set_default_prefixes({...})` (`None` clears it); `DEFAULT_PREFIXES` shows the current ones.

In [14]:
import pyoxigraph as ox

from starlayer.graph import DEFAULT_PREFIXES, set_default_prefixes

pf = StarLayerGraph(prefixes={"ex": "http://example.org/"})
pf.parse(data='@prefix dc: <http://purl.org/dc/terms/> . <http://example.org/a> dc:title "T" .', format="turtle")
print("bound:", sorted(pf.namespaces()), "| used:", pf.used_prefixes())
print(pf.expand_curie("ex:alice"), pf.compute_qname(ox.NamedNode("http://example.org/alice")), pf.qname_term(ox.NamedNode("http://example.org/alice")))
try:
    pf.compute_qname(ox.NamedNode("http://unknown.example/x#y"), generate=False)
except KeyError as error:
    print("KeyError:", error)
print("unbind dc:", pf.unbind("dc"), "->", sorted(pf.namespaces()))

set_default_prefixes({"ex": "http://example.org/"})
print("a new graph starts with:", sorted(StarLayerGraph().namespaces()), "| DEFAULT_PREFIXES:", dict(DEFAULT_PREFIXES))
set_default_prefixes(None)
print("after clearing:", sorted(StarLayerGraph().namespaces()))

bound: [('dc', 'http://purl.org/dc/terms/'), ('ex', 'http://example.org/')] | used: [('dc', 'http://purl.org/dc/terms/'), ('ex', 'http://example.org/')]
<http://example.org/alice> ('ex', 'http://example.org/', 'alice') ex:alice
KeyError: "no prefix is bound for 'http://unknown.example/x#'"
unbind dc: True -> [('ex', 'http://example.org/')]
a new graph starts with: [('ex', 'http://example.org/')] | DEFAULT_PREFIXES: {'ex': 'http://example.org/'}
after clearing: []


## 13. `owl:imports`

`graph.owl_imports()` lists the IRIs a graph imports (it fetches nothing), and `graph.has_import_locally(iri)` says whether the graph already has that ontology. StarLayer never fetches an import on its own: `graph.resolve_imports(loader=...)` follows the imports transitively with a `loader` you supply, a function from an IRI to a graph (or `None`), so you decide how, and whether, a document is fetched. It returns a new graph with the imports merged (`mode="delta"` gives only what was added, `mode="in-place"` adds to the graph); an import that is already there or that the loader cannot give is skipped. `can_resolve_import(iri, loader=...)` asks the loader whether it would give a graph for an IRI. The same `loader` idea is `shapes_graph_loader=` of `validate()` ([4.e](04e-shacl-profiling.ipynb)).

In [15]:
from starlayer.graph import can_resolve_import

BASE = ox.NamedNode("http://example.org/base")
onto = StarLayerGraph()
onto.parse(data="<http://example.org/onto> <http://www.w3.org/2002/07/owl#imports> <http://example.org/base> .", format="turtle")

def loader(iri):  # a real loader would read a file or fetch a URL
    if iri == BASE:
        found = StarLayerGraph()
        found.parse(data="<http://example.org/base> a <http://www.w3.org/2002/07/owl#Ontology> .", format="turtle")
        return found
    return None

print("imports:", onto.owl_imports(), "| has it:", onto.has_import_locally(BASE))
print("loader can give base:", can_resolve_import(BASE, loader=loader), "| another:", can_resolve_import(ox.NamedNode("http://example.org/none"), loader=loader))
merged = onto.resolve_imports(loader=loader)
print(len(onto), "triples before,", len(merged), "after; the merged graph has it:", merged.has_import_locally(BASE))

imports: (<NamedNode value=http://example.org/base>,) | has it: False
loader can give base: True | another: False
1 triples before, 2 after; the merged graph has it: True


## 14. Blank nodes and copies

`graph.skolemize()` returns a new graph in which each blank node of a subject or object (inside triple terms too) is a skolem IRI, so it can be named from outside; `authority=` and `basepath=` choose the IRI (`bnode=` replaces only one node). `graph.de_skolemize()` turns skolem IRIs back into blank nodes. `graph.empty_like()` is a new empty graph of the same kind with the same prefixes (a dataset keeps its `default_union`), which is where a function that builds a result of the caller's kind starts. `graph.cbd(resource)` is the concise bounded description of a node: its statements and, recursively, those of its blank-node objects.

In [16]:
bn = StarLayerGraph(prefixes={"ex": "http://example.org/"})
bn.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p [ ex:q 1 ] .", format="turtle")

named = bn.skolemize(authority="http://example.org", basepath="/.well-known/genid/")
print("skolem IRIs:", all(o.value.startswith("http://example.org/.well-known/genid/") for _, _, o in named.triples((None, EXL.p, None))))
restored = named.de_skolemize()
print("blank nodes again:", all(isinstance(o, ox.BlankNode) for _, _, o in restored.triples((None, EXL.p, None))))
print("empty copy:", len(bn.empty_like()), "triples,", sorted(bn.empty_like().namespaces()))
print("cbd of ex:a:", len(bn.cbd(EXL.a)), "triples")

skolem IRIs: True
blank nodes again: True
empty copy: 0 triples, [('ex', 'http://example.org/')]
cbd of ex:a: 2 triples
